# Cox Regression Penalized

### Imports

In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored , concordance_index_ipcw
from sklearn.impute import SimpleImputer
from sksurv.util import Surv

# Clinical Data
df = pd.read_csv("./data/X_train/clinical_train.csv")
df_eval = pd.read_csv("./data/X_test/clinical_test.csv")

# Molecular Data
maf_df = pd.read_csv("./data/X_train/molecular_train.csv")
maf_eval = pd.read_csv("./data/X_test/molecular_test.csv")

target_df = pd.read_csv("./data/target_train.csv")

# TODO : Uncomment for test data ??
"""
target_df_test = pd.read_csv("./data/target_test.csv")
"""
# Preview the data
df.head()

,ID,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS
0,P132697,MSK,14.0,2.8,0.2,0.7,7.6,119.0,"46,xy,del(20)(q12)[2]/46,xy[18]"
1,P132698,MSK,1.0,7.4,2.4,0.1,11.6,42.0,"46,xx"
2,P116889,MSK,15.0,3.7,2.1,0.1,14.2,81.0,"46,xy,t(3;3)(q25;q27)[8]/46,xy[12]"
3,P132699,MSK,1.0,3.9,1.9,0.1,8.9,77.0,"46,xy,del(3)(q26q27)[15]/46,xy[5]"
4,P132700,MSK,6.0,128.0,9.7,0.9,11.1,195.0,"46,xx,t(3;9)(p13;q22)[10]/46,xx[10]"


### Cleaning

In [12]:
# Drop rows where 'OS_YEARS' is NaN if conversion caused any issues
target_df.dropna(subset=['OS_YEARS', 'OS_STATUS'], inplace=True)

# Check the data types to ensure 'OS_STATUS' is boolean and 'OS_YEARS' is numeric
print(target_df[['OS_STATUS', 'OS_YEARS']].dtypes)

# Contarget_dfvert 'OS_YEARS' to numeric if it isn’t already
target_df['OS_YEARS'] = pd.to_numeric(target_df['OS_YEARS'], errors='coerce')

# Ensure 'OS_STATUS' is boolean
target_df['OS_STATUS'] = target_df['OS_STATUS'].astype(bool)

OS_STATUS       bool
OS_YEARS     float64
dtype: object


1. Extracting number of somatic mutations 

In [6]:
# Step: Extract the number of somatic mutations per patient
# Group by 'ID' and count the number of mutations (rows) per patient
tmp = maf_df.groupby('ID').size().reset_index(name='Nmut')

# Merge with the training dataset and replace missing values in 'Nmut' with 0
df_2 = df.merge(tmp, on='ID', how='left').fillna({'Nmut': 0})

2. Selecting features and getting rid of rows with NaN values in training set

In [14]:
# Select features and target columns
features = ['BM_BLAST', 'HB', 'PLT', 'Nmut']
target = ['OS_YEARS', 'OS_STATUS']

# Merge on ID to ensure same row order
merged = df_2[['ID'] + features].merge(
    target_df[['ID'] + target], on='ID'
)

# Remove rows with missing feature values
merged = merged.dropna(subset=features)

# Build feature matrix and survival target
X = merged[features]
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', merged[target])

3. Spliting dataset

In [15]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train.head()

,BM_BLAST,HB,PLT,Nmut
561,17.0,9.0,148.0,5.0
2169,1.0,11.1,236.0,1.0
229,7.2,10.9,210.0,5.0
3110,1.0,11.2,213.0,1.0
1852,4.0,8.0,86.0,6.0


### 3. Training

1. Fitting Cox model

In [23]:
# Initialize and train the Cox Proportional Hazards model
cox = CoxPHSurvivalAnalysis()
cox.fit(X_train, y_train)

# Evaluate the model using Concordance Index IPCW
cox_cindex_train = concordance_index_ipcw(y_train, y_train, cox.predict(X_train), tau=7)[0]
cox_cindex_test = concordance_index_ipcw(y_train, y_test, cox.predict(X_test), tau=7)[0]
print(f"Cox Proportional Hazard Model Concordance Index IPCW on train: {cox_cindex_train:.2f}")
print(f"Cox Proportional Hazard Model Concordance Index IPCW on test: {cox_cindex_test:.2f}")

Cox Proportional Hazard Model Concordance Index IPCW on train: 0.69
Cox Proportional Hazard Model Concordance Index IPCW on test: 0.69


In [18]:
tmp_eval = maf_eval.groupby('ID').size().reset_index(name='Nmut')

# Merge with the training dataset and replace missing values in 'Nmut' with 0
df_eval2 = df_eval.merge(tmp_eval, on='ID', how='left').fillna({'Nmut': 0})

In [21]:
imputer = SimpleImputer(strategy='median')
imputer.fit(X_train[features],y_train)
df_eval2[features] = imputer.transform(df_eval2[features])

prediction_on_test_set = cox.predict(df_eval2.loc[:, features])

In [22]:
prediction_on_test_set

array([ 0.67042862, -0.6002542 , -1.7300624 , ..., -1.73223816,
       -1.4933833 , -1.61281073])

In [57]:
submission = pd.Series(prediction_on_test_set, index=df_eval['ID'], name='risk_score')

In [58]:
submission.to_csv('./submission/cox_ridge_submission.csv')

In [59]:
submission.head()

ID
KYW1    0.670429
KYW2   -0.600254
KYW3   -1.730062
KYW4    0.326759
KYW5   -1.124396
Name: risk_score, dtype: float64